# 03 HLS物候与植被作物比较

1. 落叶阔叶林：QA、日序列、Chen SG/Whittaker/DL、不同日期定义。
2. 常绿林检验高背景/弱振幅预期；草地与落叶林比较。
3. 华北冬小麦春季与夏玉米：经验窗口起步，允许据曲线修订。
4. 东北玉米、水稻、大豆比较；同作物跨区域比较。
5. 小作业：冬小麦越冬前秋季SOS。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from scipy.optimize import least_squares
from scipy.special import expit
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/vegetation_lab/hls_phenology_io.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from vegetation_lab import chen_sg_filter as chen
from vegetation_lab import hls_phenology_io as hio
from vegetation_lab import hls_teaching_support as lab

DATA = ROOT / "data/hls"
RUN_DIR = lab.new_run(ROOT)
FIG_DIR = RUN_DIR / "figures"
SG = dict(trend_window=51, trend_polyorder=2, sg_window=31, sg_polyorder=2, iterations=10)
WH_LAMBDA = 1000
WINDOWS = {
    "DB_FOREST": ("2024-01-01", "2024-12-31"),
    "EB_FOREST": ("2024-01-01", "2024-12-31"),
    "STEPPE": ("2024-01-01", "2024-12-31"),
    "NC_WHEAT": ("2024-02-15", "2024-06-30"),
    "NC_MAIZE": ("2024-06-15", "2024-11-15"),
    "NE_MAIZE": ("2024-03-01", "2024-11-30"),
    "NE_RICE": ("2024-03-01", "2024-11-30"),
    "NE_SOY": ("2024-03-01", "2024-11-30"),
    "JS_RICE": ("2024-05-01", "2024-11-30"),
}
display(hio.inventory(DATA))
print("输出目录：", RUN_DIR)
print("本次SG脚本：", chen.__file__)
sample = lab.load_region(ROOT, "DB_FOREST")
display(pd.DataFrame([sample.attrs]))

## 1. 落叶阔叶林：从观测到物候

### 1.1 Fmask怎样保留观测？

In [ ]:
def qa_mask(ndvi, fmask, exclude_adjacent=True):
    present = (np.isfinite(ndvi) & (ndvi >= -1) & (ndvi <= 1)
               & np.isfinite(fmask) & (fmask >= 0) & (fmask < 255)
               & (fmask == np.rint(fmask)))
    qa = np.where(present, fmask, 0).astype("uint16")
    cloud = (qa & (1 << 1)) != 0
    adjacent = (qa & (1 << 2)) != 0
    shadow = (qa & (1 << 3)) != 0
    snow = (qa & (1 << 4)) != 0
    # Water is not removed: the same input convention is used for rice.
    return present & ~cloud & ~shadow & ~snow & (~adjacent if exclude_adjacent else True)

In [ ]:
point = lab.point_record(sample, qa_mask)
display(point.head(12))
print("有效逐景观测：", point.qc.notna().sum(), "/", len(point))
lab.show_methods(point, pd.DatetimeIndex([]), {}, FIG_DIR / "01_raw_qa.png")

### 1.2 同日合并、日时间轴与插值

In [ ]:
def daily_observations(dates, values, start, end):
    dates = pd.DatetimeIndex(dates).normalize()
    grid = pd.date_range(start, end, freq="D")
    daily = np.full((len(grid), *np.shape(values)[1:]), np.nan)
    for date in dates.unique().sort_values():
        if grid[0] <= date <= grid[-1]:
            samples = np.ma.masked_invalid(np.asarray(values)[dates == date])
            daily[(date-grid[0]).days] = np.ma.filled(np.ma.median(samples, axis=0), np.nan)
    return grid, daily

def interpolate_inside(observed):
    observed = np.asarray(observed, float)
    columns = observed.reshape(len(observed), -1)
    filled = np.full_like(columns, np.nan)
    time = np.arange(len(observed))
    for col in range(columns.shape[1]):
        good = np.flatnonzero(np.isfinite(columns[:, col]))
        if len(good):
            inside = (time >= good[0]) & (time <= good[-1])
            filled[inside, col] = np.interp(time[inside], good, columns[good, col])
    return filled.reshape(observed.shape)

In [ ]:
START, END = WINDOWS["DB_FOREST"]
grid, observed = daily_observations(point.date, point.qc.to_numpy(), START, END)
linear = interpolate_inside(observed)
display(pd.DataFrame({"date": grid, "真实同日观测": observed, "日网格插值": linear}).iloc[::20])

### 1.3 指定脚本的Chen迭代SG

脚本已从项目根目录迁入 `src/vegetation_lab/chen_sg_filter.py`，内容未修改。

处理链：初始趋势 → 低于趋势的点降权 → 初始向上修正 → 两轮SG → 按相邻三轮加权误差选择更新。

$$w_t=1-\mathbf{1}_{y_t<T_t}\frac{|y_t-T_t|}{\max_t|y_t-T_t|},\qquad
F_k=\sum_t w_t|s_t^{(k)}-y_t|.$$

分母为0时脚本设权重为1。

```python
    # Initial trend, asymmetric weights, and the first upward corrections.
    trend_series = _sg_filter(values, trend)
    weights = _weights(values, trend_series)
    current = np.maximum(values, trend_series)

    # The original script performs two SG passes before entering the loop.
    fk_values: list[np.ndarray] = []
    filtered = _sg_filter(current, sg)
    fk_values.append(np.sum(np.abs(filtered - values) * weights, axis=0))
    current = np.maximum(filtered, values)

    filtered = _sg_filter(current, sg)
    fk_values.append(np.sum(np.abs(filtered - values) * weights, axis=0))
    current = np.maximum(filtered, values)

    # The EE sequence is ee.List.sequence(2, 10), inclusive.
    for index in range(2, iterations + 1):
        filtered = _sg_filter(current, sg)
        fk_values.append(np.sum(np.abs(filtered - values) * weights, axis=0))

        previous_fk = fk_values[index - 2]
        current_fk = fk_values[index - 1]
        next_fk = fk_values[index]
        keep_previous = (previous_fk >= current_fk) & (current_fk <= next_fk)
        current = np.where(keep_previous[None, ...], current, filtered)

```

In [ ]:
def chen_daily(interpolated, settings):
    """Use the supplied script unchanged; batch only columns with the same support."""
    a = np.asarray(interpolated, float)
    flat = a.reshape(len(a), -1)
    result = np.full_like(flat, np.nan)
    spans = {}
    for col in range(flat.shape[1]):
        valid = np.flatnonzero(np.isfinite(flat[:, col]))
        if len(valid) >= 3:  # The supplied filter requires at least 3 input times.
            spans.setdefault((valid[0], valid[-1] + 1), []).append(col)
    for (left, right), columns in spans.items():
        for offset in range(0, len(columns), 256):
            cols = columns[offset:offset+256]
            result[left:right, cols] = chen.chen_sg_filter(
                flat[left:right, cols], nan_policy="raise", restore_nan=False, **settings)
    return result.reshape(a.shape)

In [ ]:
chen_curve = chen_daily(linear, SG)
display(pd.DataFrame({"趋势系数": pd.Series(chen.make_sg_coeffs(SG["trend_window"], SG["trend_polyorder"])),
                      "迭代系数": pd.Series(chen.make_sg_coeffs(SG["sg_window"], SG["sg_polyorder"]))}))
lab.show_methods(point, grid, {"线性插值": linear, "Chen迭代SG": chen_curve}, FIG_DIR / "02_chen.png")

### 1.4 Whittaker：真实观测权重与平滑惩罚

$$\min_z\sum_i w_i(y_i-z_i)^2+\lambda\sum_i(\Delta^2z_i)^2.$$

真实观测日权重1，缺测日权重0。

In [ ]:
def whittaker_daily(observed, lam):
    n = len(observed)
    weights = np.isfinite(observed).astype(float)
    if n < 3 or weights.sum() < 2:
        return np.full(n, np.nan)
    D = diags([np.ones(n-2), -2*np.ones(n-2), np.ones(n-2)], [0, 1, 2], shape=(n-2, n))
    W = diags(weights)
    curve = spsolve((W + lam * D.T @ D).tocsc(), weights * np.nan_to_num(observed))
    support = np.flatnonzero(weights)
    curve[:support[0]] = np.nan
    curve[support[-1]+1:] = np.nan
    return curve

In [ ]:
wh_curve = whittaker_daily(observed, WH_LAMBDA)

### 1.5 Double Logistic：拟合曲线，也是参数化物候方法

$$VI(t)=b+\frac{a}{1+e^{-k_1(t-t_1)}}-\frac{a}{1+e^{-k_2(t-t_2)}}.$$

In [ ]:
def double_logistic(t, baseline, amplitude, rise, duration, k_rise, k_fall):
    return baseline + amplitude * (expit(k_rise*(t-rise)) - expit(k_fall*(t-rise-duration)))

def fit_dl_daily(observed):
    t = np.arange(len(observed), dtype=float)
    valid = np.isfinite(observed)
    if not valid.any() or len(t) < 3:
        return None, np.full(len(t), np.nan)
    y = observed[valid]
    span = float(len(t)-1)
    initial = [np.min(y), max(np.ptp(y), .01), span*.25, span*.5, .05, .05]
    result = least_squares(lambda p: double_logistic(t[valid], *p)-y, initial,
                           bounds=([-1.5, 0, 0, 1, .001, .001], [1.5, 3, span, 2*span, 1, 1]),
                           x_scale="jac", max_nfev=1500)
    curve = double_logistic(t, *result.x)
    return result, curve

def parameter_dates(params, fitted):
    baseline, amplitude, rise, duration, k_rise, k_fall = params
    SOS = rise
    EOS = rise + duration
    POS = float(np.argmax(fitted))
    return np.array([SOS, POS, EOS, EOS-SOS])

In [ ]:
dl_fit, dl_curve = fit_dl_daily(observed)
params = dl_fit.x if dl_fit is not None else np.full(6, np.nan)
dl_events = parameter_dates(params, dl_curve) if dl_fit is not None else np.full(4, np.nan)
display(pd.DataFrame({"参数": ["b", "a", "t1", "duration", "k1", "k2"], "拟合值": params}))
print(dl_fit.message if dl_fit is not None else "没有可拟合的观测")
display(pd.DataFrame([lab.dates_row("DL参数法", dl_events, START)]))
curves = {"线性插值": linear, "Chen SG": chen_curve, "Whittaker": wh_curve, "DL": dl_curve}
lab.show_methods(point, grid, curves, FIG_DIR / "03_methods.png")

### 1.6 峰值、两侧基线与动态阈值


In [ ]:
def threshold_dates(curve, ratio=.2):
    t = np.flatnonzero(np.isfinite(curve))
    if not len(t):
        return np.full(4, np.nan)
    y = np.asarray(curve)[t]
    peak = int(np.argmax(y))
    base_left, base_right = np.min(y[:peak+1]), np.min(y[peak:])
    up_level = base_left + ratio*(y[peak]-base_left)
    down_level = base_right + ratio*(y[peak]-base_right)
    up = np.flatnonzero((y[:-1] < up_level) & (y[1:] >= up_level))
    down = np.flatnonzero((y[:-1] >= down_level) & (y[1:] < down_level))
    up, down = up[up < peak], down[down >= peak]

    def crossing(candidates, level, last):
        if not len(candidates):
            return np.nan
        i = candidates[-1] if last else candidates[0]
        return t[i] + (level-y[i])/(y[i+1]-y[i])*(t[i+1]-t[i])

    SOS = crossing(up, up_level, True)
    EOS = crossing(down, down_level, False)
    return np.array([SOS, float(t[peak]), EOS, EOS-SOS])

In [ ]:
chen20 = threshold_dates(chen_curve, .2)
chen50 = threshold_dates(chen_curve, .5)
peak = int(np.nanargmax(chen_curve))
left_base, right_base = np.nanmin(chen_curve[:peak+1]), np.nanmin(chen_curve[peak:])
display(pd.DataFrame([{"峰值": chen_curve[peak], "峰前基线": left_base, "峰后基线": right_base,
                      "上升20%": left_base+.2*(chen_curve[peak]-left_base),
                      "下降20%": right_base+.2*(chen_curve[peak]-right_base)}]))
lab.show_events(grid, chen_curve, {"20%": chen20, "50%": chen50}, FIG_DIR / "04_threshold.png",
                "落叶林：同一Chen SG曲线的两种阈值", ratio=.2)

### 1.7 同一次DL拟合：t1/t2与50%阈值

In [ ]:
dl50 = threshold_dates(dl_curve, .5)
dl20 = threshold_dates(dl_curve, .2)
dl_compare = pd.DataFrame([lab.dates_row("DL参数法", dl_events, START), lab.dates_row("DL+50%", dl50, START)])
display(dl_compare)
display(pd.DataFrame([dl50-dl_events], columns=lab.METRICS, index=["DL50减参数法（天）"]))
lab.show_events(grid, dl_curve, {"t1/t2": dl_events, "50%": dl50}, FIG_DIR / "05_dl_compare.png",
                "同一次DL拟合，两种物候定义", ratio=.5)

### 1.8 速率特征与两类控制比较

速率法定位峰前最大上升速率、峰后最大下降速率。二阶特征取最大上升速率之前和最大下降速率之后的正加速度极值；
导数结果标为特征日期与特征间隔。

In [ ]:
def derivative_dates(curve):
    valid = np.flatnonzero(np.isfinite(curve))
    if len(valid) < 3:
        return np.full(4, np.nan), np.full(4, np.nan)
    y = curve[valid]
    peak = int(np.argmax(y))
    d1 = np.gradient(y, valid)
    d2 = np.gradient(d1, valid)
    rise = int(np.argmax(d1[:peak+1]))
    fall = peak + int(np.argmin(d1[peak:]))
    rise2 = int(np.argmax(d2[:rise+1]))
    fall2 = fall + int(np.argmax(d2[fall:]))
    first = np.array([valid[rise], valid[peak], valid[fall], valid[fall]-valid[rise]], float)
    second = np.array([valid[rise2], valid[peak], valid[fall2], valid[fall2]-valid[rise2]], float)
    return first, second

In [ ]:
rate_dates, acceleration_dates = derivative_dates(wh_curve)
method_events = {"Chen SG+20%": chen20, "Chen SG+50%": chen50,
                 "Whittaker+20%": threshold_dates(wh_curve, .2), "Whittaker+50%": threshold_dates(wh_curve, .5),
                 "DL参数法": dl_events, "DL+20%": dl20, "DL+50%": dl50}
method_table = pd.DataFrame([lab.dates_row(name, dates, START) for name, dates in method_events.items()])
display(method_table)
display(pd.DataFrame([lab.dates_row("Whittaker速率特征（LOS列为特征间隔）", rate_dates, START),
                      lab.dates_row("Whittaker二阶特征（LOS列为特征间隔）", acceleration_dates, START)]))
lab.show_derivatives(grid, wh_curve, FIG_DIR / "06_derivatives.png")
method_table.to_csv(RUN_DIR / "method_comparison.csv", index=False, encoding="utf-8-sig")

### 1.9 参数记录：每次只改一个设置

修改下方SG迭代窗口或Whittaker的λ之一，再运行本单元记录日期和曲线变化。不要同时更改两项。

In [ ]:
TRIAL_SG = dict(SG)
TRIAL_LAMBDA = WH_LAMBDA
# 例如下一次修改 TRIAL_SG["sg_window"] = 15；保持趋势窗口和其他设置不变。
trial_sg = chen_daily(linear, TRIAL_SG)
trial_wh = whittaker_daily(observed, TRIAL_LAMBDA)
display(pd.DataFrame([lab.dates_row("本次Chen SG+20%", threshold_dates(trial_sg, .2), START),
                      lab.dates_row("本次Whittaker+20%", threshold_dates(trial_wh, .2), START)]))
print(TRIAL_SG, "lambda=", TRIAL_LAMBDA)

## 2. 代表像元与完整ROI：天然植被比较

In [ ]:
def season_point(record, start, end, settings):
    days, obs = daily_observations(record.date, record.qc.to_numpy(), start, end)
    interpolated = interpolate_inside(obs)
    curve = chen_daily(interpolated, settings)
    events = threshold_dates(curve, .2)
    return days, curve, events

points = {"DB_FOREST": point}
sample_rows = [{"ROI": "DB_FOREST", **sample.attrs}]
for roi in ["EB_FOREST", "STEPPE", "NC_ROT", "NE_MAIZE", "NE_RICE", "NE_SOY", "JS_RICE"]:
    loaded = lab.load_region(ROOT, roi)
    points[roi] = lab.point_record(loaded, qa_mask)
    sample_rows.append({"ROI": roi, **loaded.attrs})
display(pd.DataFrame(sample_rows)[["ROI", "center_lon", "center_lat", "sample_source", "excluded_other_grid_scenes"]])
baseline = {}
point_rows = []
for key, (start, end) in WINDOWS.items():
    roi = "NC_ROT" if key.startswith("NC_") else key
    days, curve, events = season_point(points[roi], start, end, SG)
    baseline[key] = (days, curve, events)
    point_rows.append({"ROI": key, **lab.dates_row("Chen SG+20%", events, start)})
point_table = pd.DataFrame(point_rows)
point_table.to_csv(RUN_DIR / "representative_dates.csv", index=False, encoding="utf-8-sig")

In [ ]:
def roi_statistics(cube, label, start, end):
    rows = []
    for i, metric in enumerate(["SOS", "POS", "EOS", "LOS"]):
        values = cube[i][np.isfinite(cube[i])]
        q1, median, q3 = np.quantile(values, [.25, .5, .75]) if len(values) else [np.nan]*3
        rows.append(dict(ROI=label, window_start=start, window_end=end,
                         method="Chen SG + 20%", metric=metric, median=median,
                         Q1=q1, Q3=q3, IQR=q3-q1, N=len(values)))
    return pd.DataFrame(rows)

In [ ]:
algorithms = [qa_mask, daily_observations, interpolate_inside, chen_daily, threshold_dates, roi_statistics]
lab.save_configuration(ROOT, RUN_DIR, {"windows": WINDOWS, "SG": SG, "Whittaker_lambda": WH_LAMBDA,
                                      "sample_locations": [{"ROI": r["ROI"], "lon": r["center_lon"], "lat": r["center_lat"]} for r in sample_rows]},
                       algorithms+[whittaker_daily, double_logistic, fit_dl_daily, parameter_dates])
maps, roi_stats = lab.run_regions(ROOT, WINDOWS, SG, algorithms, RUN_DIR)
display(roi_stats)
lab.show_maps(maps["DB_FOREST"], FIG_DIR / "07_db_map.png", "落叶阔叶林 Chen SG+20%")

### 2.· 常绿阔叶林

In [ ]:
keys = ["DB_FOREST", "EB_FOREST"]
lab.show_series({k: points[k] for k in keys}, {k: baseline[k][:2] for k in keys},
                "天然植被：真实数据是否支持预期？", FIG_DIR / "08_forests.png")
lab.show_distributions(maps, keys, FIG_DIR / "09_forests_roi.png", "两类森林的完整ROI物候分布")
display(roi_stats[roi_stats.ROI.isin(keys)])

### 2.4 草地与落叶阔叶林

In [ ]:
keys = ["DB_FOREST", "STEPPE"]
lab.show_series({k: points[k] for k in keys}, {k: baseline[k][:2] for k in keys},
                "草地与落叶林：物候期与长度", FIG_DIR / "10_grass_forest.png")
lab.show_distributions(maps, keys, FIG_DIR / "11_grass_roi.png", "完整ROI分布")
display(lab.comparison_table(roi_stats, [("DB_FOREST", "STEPPE")]))

## 3. 华北轮作：经验窗口、分季计算与学生判断

### 3.1 展示窗口不同于提取窗口

In [ ]:
rotation = points["NC_ROT"]
rotation_view = rotation[(rotation.date >= "2023-09-01") & (rotation.date < "2024-12-01")]
lab.show_series({"NC_ROT": rotation_view}, {}, "华北轮作：秋季背景、春季小麦与夏玉米", FIG_DIR / "12_rotation.png",
                windows={"小麦春季经验窗口": WINDOWS["NC_WHEAT"], "夏玉米经验窗口": WINDOWS["NC_MAIZE"]})
display(point_table[point_table.ROI.isin(["NC_WHEAT", "NC_MAIZE"])])
display(roi_stats[roi_stats.ROI.isin(["NC_WHEAT", "NC_MAIZE"])])
for key in ["NC_WHEAT", "NC_MAIZE"]:
    days, curve, events = baseline[key]
    lab.show_events(days, curve, {lab.LABELS[key]: events}, FIG_DIR / f"13_{key}.png", lab.LABELS[key])
    lab.show_maps(maps[key], FIG_DIR / f"14_{key}_map.png", lab.LABELS[key])

### 3.2 判断并修订经验窗口

小麦窗口开始过晚可能丢掉返青，结束过晚可能混入夏玉米。

In [ ]:
WHEAT_WINDOW = WINDOWS["NC_WHEAT"]  # 可以替换为有依据的新起止日期
MAIZE_WINDOW = WINDOWS["NC_MAIZE"]
WINDOW_REASON = "默认经验窗口；请观察曲线后填写保留或修改理由"
window_rows = []
for key, changed in [("NC_WHEAT", WHEAT_WINDOW), ("NC_MAIZE", MAIZE_WINDOW)]:
    days, curve, events = season_point(rotation, *changed, SG)
    original_start = pd.Timestamp(WINDOWS[key][0])
    changed_start = pd.Timestamp(changed[0])
    delta = events-baseline[key][2]
    delta[:3] += (changed_start-original_start).days
    window_rows.append({"season": key, "default_window": str(WINDOWS[key]), "changed_window": str(changed),
                        "reason": WINDOW_REASON, "SOS_change_days": delta[0], "POS_change_days": delta[1],
                        "EOS_change_days": delta[2], "LOS_change_days": delta[3],
                        **lab.dates_row("调整后Chen SG+20%", events, changed[0])})
display(pd.DataFrame(window_rows))
pd.DataFrame(window_rows).to_csv(RUN_DIR / "window_experiment.csv", index=False, encoding="utf-8-sig")

## 4. 东北玉米、水稻、大豆

In [ ]:
keys = ["NE_MAIZE", "NE_RICE", "NE_SOY"]
lab.show_series({k: points[k] for k in keys}, {k: baseline[k][:2] for k in keys},
                "东北三作物：代表像元曲线", FIG_DIR / "15_ne_crops.png")
display(point_table[point_table.ROI.isin(keys)])
display(roi_stats[roi_stats.ROI.isin(keys)])
lab.show_distributions(maps, keys, FIG_DIR / "16_ne_roi.png", "东北三作物完整ROI")

## 5. 同作物跨区域

### 5.1 东北玉米与华北夏玉米


In [ ]:
keys = ["NE_MAIZE", "NC_MAIZE"]
lab.show_series({"NE_MAIZE": points["NE_MAIZE"], "NC_MAIZE": rotation_view},
                {k: baseline[k][:2] for k in keys}, "玉米的区域与种植制度差异", FIG_DIR / "17_maize_regions.png")
lab.show_distributions(maps, keys, FIG_DIR / "18_maize_roi.png", "玉米：完整ROI比较")
display(lab.comparison_table(roi_stats, [("NE_MAIZE", "NC_MAIZE")]))

### 5.2 东北水稻与江苏水稻

In [ ]:
keys = ["NE_RICE", "JS_RICE"]
lab.show_series({k: points[k] for k in keys}, {k: baseline[k][:2] for k in keys},
                "水稻跨区域比较", FIG_DIR / "19_rice_regions.png")
lab.show_distributions(maps, keys, FIG_DIR / "20_rice_roi.png", "水稻：完整ROI比较")
pairs = [("DB_FOREST", "STEPPE"), ("NE_MAIZE", "NC_MAIZE"), ("NE_RICE", "JS_RICE")]
regional_comparison = lab.comparison_table(roi_stats, pairs)
display(regional_comparison)
regional_comparison.to_csv(RUN_DIR / "regional_comparison.csv", index=False, encoding="utf-8-sig")

## 6. 小作业：冬小麦越冬前的SOS

用上一年秋季观测选择窗口，提取秋季增绿SOS，解释它与春季返青SOS的区别。

In [ ]:
AUTUMN_WINDOW = ("2023-09-01", "2023-12-31")
autumn_days, autumn_curve, autumn_events = season_point(rotation, *AUTUMN_WINDOW, SG)
SOS_wheat_autumn = lab.dates_row("秋季SOS作业起点", autumn_events, AUTUMN_WINDOW[0])["SOS"]
SOS_wheat_spring = lab.dates_row("春季", baseline["NC_WHEAT"][2], WINDOWS["NC_WHEAT"][0])["SOS"]
display(pd.DataFrame([{"SOS_wheat_autumn": SOS_wheat_autumn, "SOS_wheat_spring": SOS_wheat_spring,
                      "两次SOS间隔(不是LOS)": (SOS_wheat_spring-SOS_wheat_autumn).total_seconds()/86400}]))
lab.show_events(autumn_days, autumn_curve, {"秋季窗口特征": autumn_events}, FIG_DIR / "21_autumn.png",
                "冬小麦秋季：检查SOS交叉是否合理", ratio=.2)